In [ ]:
# Create local Ollama LLM and embedding model
from langchain_ollama import ChatOllama, OllamaEmbeddings
# Make sure these models are available locally:
#   ollama pull llama3.1
#   ollama pull nomic-embed-text
LLM_MODEL = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"
# Local chat model for answering, routing, and generation.
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)
# Local embedding model for converting text chunks into vectors.
embedding_model = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)
print("Ollama LLM and embedding model initialized.")
print("LLM model:", LLM_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
response = llm.invoke("Reply with exactly: Ollama is working")
print(response.content)

In [8]:
from langchain_core.documents import Document
# Synthetic company onboarding knowledge base for the exercise.
# In a real project, these documents could come from PDFs, SharePoint, Confluence, Notion, websites, or HR systems.
documents = [
    Document(
        page_content="""
        Remote Work Policy:
        Employees are allowed to work remotely up to two days per week.
        Remote work must be coordinated with the direct manager.
        Employees must remain available during core working hours from 10:00 AM to 4:00 PM.
        Fully remote work requires approval from the department head.
        """,
        metadata={"source": "remote_work_policy"}
    ),
    Document(
        page_content="""
        Leave Policy:
        Full-time employees receive 22 annual leave days per year.
        Sick leave must be reported to the direct manager as early as possible.
        For sick leave longer than two consecutive days, a medical certificate is required.
        Unused annual leave can be carried forward for a maximum of five days.
        """,
        metadata={"source": "leave_policy"}
    ),
    Document(
        page_content="""
        Business Travel Policy:
        Business travel must be approved by the employee's direct manager and the finance department.
        Employees should submit travel requests at least ten working days before the planned trip.
        Economy class is required for flights shorter than six hours.
        Hotel bookings should follow the approved corporate travel list.
        """,
        metadata={"source": "business_travel_policy"}
    ),
Document(
        page_content="""
        Password Reset Policy:
        Employees who forget their password should use the self-service password reset portal.
        If the portal does not work, they should contact the IT helpdesk.
        Passwords must not be shared with colleagues or managers.
        Multi-factor authentication is required for all company systems.
        """,
        metadata={"source": "password_policy"}
    ),
    Document(
        page_content="""
        Expense Reimbursement Policy:
        Employees must submit reimbursement claims within 30 days of the expense date.
        Receipts are required for all expenses above 25 dollars.
        Claims must be approved by the direct manager and finance team.
        """,
        metadata={"source": "expense_policy"}
    ),
    Document(
        page_content="""
        Training and Development Policy:
        Employees may request professional training related to their role.
        Training requests must include the course name, provider, cost, and expected business benefit.
        Requests require approval from the direct manager and HR.
        """,
        metadata={"source": "training_policy"}
    )
]
print(f"Loaded {len(documents)} source documents.")

Loaded 6 source documents.


In [10]:
# =========================================================
# 1. Split documents into chunks
# =========================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60
)
chunks = text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")
print("\nPreview chunk:\n", chunks[0].page_content)
print("Metadata:", chunks[0].metadata)
# =========================================================
# 2. Create FAISS vector store and retriever
# =========================================================
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# =========================================================
# 3. Helper function to format retrieved chunks
# =========================================================
def format_docs(docs):
    """Convert retrieved documents into a single context string with source names."""
    formatted_text = ""
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content.strip()
        formatted_text += f"\nSource: {source}\nContent: {content}\n"
    return formatted_text
# =========================================================
# 4. Test retrieval before using the LLM
# =========================================================
test_question = "Who approves business travel?"
retrieved_docs = retriever.invoke(test_question)
print("Question:", test_question)
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content.strip())
    print("Source:", doc.metadata.get("source"))
    # =========================================================
# 5. Build Classic RAG chain
# =========================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful company onboarding assistant.
Answer the user's question using only the context provided below.
If the answer is not in the context, say:
"I do not have enough information in the provided documents to answer this question."
Context:
{context}
Question:
{question}
Answer:
""")
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
# =========================================================
# 6. Test Classic RAG
# =========================================================
classic_questions = [
    "How many days per week can employees work remotely?",
    "Who approves business travel?",
    "How do I request a laptop?",
    "What is the company's bonus policy?"
]
for q in classic_questions:
    print("\n" + "="*80)
    print("QUESTION:", q)
    print("ANSWER:")
    print(rag_chain.invoke(q))


In [ ]:
# =========================================================
# 7. Build Agentic RAG with LangGraph
# =========================================================

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


# ---------------------------------------------------------
# 1. Define shared state
# ---------------------------------------------------------

class AgentState(TypedDict):
    """Shared state passed between LangGraph nodes."""
    question: str
    route: str
    context: str
    answer: str


# ---------------------------------------------------------
# 2. Router Prompt
# ---------------------------------------------------------

router_prompt = ChatPromptTemplate.from_template("""
You are a routing assistant.

Decide whether the user's question requires searching the company policy documents.

Return only one word:
- retrieve
- direct

Choose "retrieve" if the question is about:
remote work, leave, vacation, sick leave, travel, equipment, password, IT,
approvals, company policies, onboarding processes, expenses, reimbursement,
training, development.

Choose "direct" if the question is:
a greeting, small talk, or a general question not requiring company documents.

User question:
{question}
""")


# ---------------------------------------------------------
# 3. Router Node
# ---------------------------------------------------------

def route_question(state: AgentState):
    """LLM router: decide whether to retrieve or answer directly."""

    router_chain = router_prompt | llm | StrOutputParser()

    route = router_chain.invoke({
        "question": state["question"]
    }).strip().lower()

    if route not in ["retrieve", "direct"]:
        route = "retrieve"  # safe fallback

    return {"route": route}


# ---------------------------------------------------------
# 4. Conditional Edge
# ---------------------------------------------------------

def decide_next_step(
    state: AgentState
) -> Literal["retrieve", "direct"]:
    """Conditional edge function used by LangGraph."""

    return "retrieve" if state["route"] == "retrieve" else "direct"


# ---------------------------------------------------------
# 5. Retrieval Node
# ---------------------------------------------------------

def retrieve_context(state: AgentState):
    """Retrieve relevant chunks from the vector store."""

    docs = retriever.invoke(state["question"])

    context = format_docs(docs)

    return {"context": context}


# ---------------------------------------------------------
# 6. RAG Answer Prompt
# ---------------------------------------------------------

agentic_rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful company onboarding assistant.

Use only the context below to answer the user's question.

Include the source name when possible.

If the answer is not in the context, say:
"I do not have enough information in the provided documents to answer this question."

Context:
{context}

Question:
{question}

Answer:
""")


# ---------------------------------------------------------
# 7. RAG Answer Node
# ---------------------------------------------------------

def generate_rag_answer(state: AgentState):
    """Generate grounded answer using retrieved context."""

    chain = agentic_rag_prompt | llm | StrOutputParser()

    answer = chain.invoke({
        "question": state["question"],
        "context": state["context"]
    })

    return {"answer": answer}


# ---------------------------------------------------------
# 8. Direct Answer Prompt
# ---------------------------------------------------------

direct_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

The user asked a question that does not require company policy retrieval.

Answer briefly and helpfully.

User question:
{question}

Answer:
""")


# ---------------------------------------------------------
# 9. Direct Answer Node
# ---------------------------------------------------------

def generate_direct_answer(state: AgentState):
    """Generate direct answer without retrieval."""

    chain = direct_prompt | llm | StrOutputParser()

    answer = chain.invoke({
        "question": state["question"]
    })

    return {
        "context": "",
        "answer": answer
    }


# ---------------------------------------------------------
# 10. Build LangGraph
# ---------------------------------------------------------

workflow = StateGraph(AgentState)

workflow.add_node("router", route_question)
workflow.add_node("retrieve", retrieve_context)
workflow.add_node("rag_answer", generate_rag_answer)
workflow.add_node("direct_answer", generate_direct_answer)

workflow.add_edge(START, "router")

workflow.add_conditional_edges(
    "router",
    decide_next_step,
    {
        "retrieve": "retrieve",
        "direct": "direct_answer"
    }
)

workflow.add_edge("retrieve", "rag_answer")
workflow.add_edge("rag_answer", END)

workflow.add_edge("direct_answer", END)


# ---------------------------------------------------------
# 11. Compile Graph
# ---------------------------------------------------------

agentic_rag_graph = workflow.compile()

print("Agentic RAG graph compiled successfully.")


# ---------------------------------------------------------
# 12. Display Graph
# ---------------------------------------------------------

from IPython.display import Image, display

try:
    display(
        Image(
            agentic_rag_graph
            .get_graph()
            .draw_mermaid_png()
        )
    )

except Exception as error:
    print("PNG rendering is unavailable.")
    print("\nMermaid definition:")
    print(agentic_rag_graph.get_graph().draw_mermaid())
    print(f"\nRendering error: {error}")

In [ ]:
# =========================================================
# 8. Test Agentic RAG
# =========================================================
def run_agentic_rag(question: str):
    """Helper function to run the LangGraph agent."""
    result = agentic_rag_graph.invoke({
        "question": question,
        "route": "",
        "context": "",
        "answer": ""
    })
    print("\n" + "="*80)
    print("QUESTION:", question)
    print("ROUTE:", result["route"])
    print("ANSWER:")
    print(result["answer"])
    return result
agentic_questions = [
    "Hi, what can you help me with?",
    "can i use my phone during work ?",
    "Who approves reimbursement claims?",
    "What should I do if I forget my password?",
    "What is the company cafeteria menu?"
]
for q in agentic_questions:
    run_agentic_rag(q)

In [ ]:
# =========================================================
# 9. Optional: show retrieved context for a policy question
# =========================================================
result = run_agentic_rag("How do I request professional training?")
print("\nRetrieved context used by the agent:\n")
print(result["context"])
